# CI/CD & Monitoring — Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

**Exercise 1: CI/CD/CT Distinctions (Conceptual).** Identify whether each scenario describes CI, CD, or CT.

In [ ]:
answers = {
    1: ("CT", "Retraining on fresh data on a schedule = Continuous Training"),
    2: ("CI", "PR triggers automated build + tests = Continuous Integration"),
    3: ("CD", "After tests pass, artifact pushed to staging = Continuous Delivery"),
    4: ("CT", "Drift alert triggers automatic retraining = Continuous Training"),
    5: ("CI", "Automated tests on every PR = Continuous Integration"),
}

for num, (category, reason) in answers.items():
    print(f"{num}. {category}: {reason}")

**Exercise 2: Workflow Simulator (Coding).** Create an enhanced workflow simulator that stops at first failure.

In [ ]:
import time

def run_enhanced_workflow(steps):
    """Run workflow steps, stopping at first failure."""
    log = []
    total_time = 0.0
    failed_step = None

    for step in steps:
        name = step["name"]
        passed = step["passed"]
        t = step["time_sec"]

        total_time += t

        if passed:
            log.append(f"PASS  {name} ({t:.1f}s)")
        else:
            log.append(f"FAIL  {name} ({t:.1f}s)")
            failed_step = name
            break

    return {
        "success": failed_step is None,
        "total_time": round(total_time, 1),
        "steps_run": len(log),
        "failed_step": failed_step,
        "log": log,
    }

pipeline_a = [
    {"name": "checkout", "passed": True, "time_sec": 2.3},
    {"name": "install deps", "passed": True, "time_sec": 45.1},
    {"name": "pytest", "passed": True, "time_sec": 12.8},
    {"name": "validate data", "passed": False, "time_sec": 3.2},
    {"name": "evaluate model", "passed": True, "time_sec": 8.5},
]

pipeline_b = [
    {"name": "checkout", "passed": True, "time_sec": 2.1},
    {"name": "install deps", "passed": True, "time_sec": 43.7},
    {"name": "pytest", "passed": True, "time_sec": 15.3},
    {"name": "validate data", "passed": True, "time_sec": 2.9},
    {"name": "evaluate model", "passed": True, "time_sec": 9.1},
]

for name, pipeline in [("A", pipeline_a), ("B", pipeline_b)]:
    result = run_enhanced_workflow(pipeline)
    print(f"Pipeline {name}: success={result['success']}, time={result['total_time']}s, "
          f"steps={result['steps_run']}, failed={result['failed_step']}")
    for entry in result["log"]:
        print(f"  {entry}")
    print()

**Exercise 3: Multi-Signal Release Gate (Coding).** Build a release gate checking tests, data, accuracy, and latency.

In [ ]:
def release_gate(tests_pass, data_valid, accuracy, latency_p95,
                 min_accuracy=0.88, max_latency=200.0):
    """Check all release conditions and collect failure reasons."""
    reasons = []

    if not tests_pass:
        reasons.append("Unit/contract tests failed")
    if not data_valid:
        reasons.append("Data validation failed")
    if accuracy < min_accuracy:
        reasons.append(f"Accuracy {accuracy:.3f} below threshold {min_accuracy}")
    if latency_p95 > max_latency:
        reasons.append(f"P95 latency {latency_p95:.0f}ms exceeds max {max_latency:.0f}ms")

    allowed = len(reasons) == 0
    return (allowed, reasons)

candidates = [
    {"name": "v1.2-baseline", "tests_pass": True, "data_valid": True,
     "accuracy": 0.91, "latency_p95": 145.0},
    {"name": "v1.3-fast", "tests_pass": True, "data_valid": True,
     "accuracy": 0.85, "latency_p95": 89.0},
    {"name": "v1.3-accurate", "tests_pass": False, "data_valid": True,
     "accuracy": 0.94, "latency_p95": 156.0},
    {"name": "v1.4-broken", "tests_pass": False, "data_valid": False,
     "accuracy": 0.82, "latency_p95": 312.0},
]

for c in candidates:
    allowed, reasons = release_gate(**{k: v for k, v in c.items() if k != "name"})
    status = "RELEASE" if allowed else "BLOCK"
    print(f"{c['name']:<20} {status}")
    for r in reasons:
        print(f"  - {r}")
    if allowed:
        print("  All gates passed")
    print()

**Exercise 4: Rollout Pattern Selection (Conceptual).** Choose the appropriate rollout pattern for each scenario.

In [ ]:
rollout_answers = {
    1: ("shadow",
        "Shadow deployment: new model processes real traffic in parallel but "
        "users never see its output. Perfect for architecture change validation "
        "without risk."),
    2: ("blue-green",
        "Blue-green: instant switchover and rollback. When you need zero-downtime "
        "deploy with instant revert capability."),
    3: ("canary",
        "Canary: small percentage of traffic, budget-conscious. Gradual rollout "
        "with minimal infrastructure overhead."),
    4: ("shadow",
        "Shadow deployment: real data flows through but results hidden from users. "
        "Safe exploration of unexpected failure modes."),
}

for num, (pattern, reason) in rollout_answers.items():
    print(f"{num}. {pattern.upper()}: {reason}\n")

**Exercise 5: Canary Decision Engine (Coding).** Implement canary promotion based on error rate comparison.

In [ ]:
import numpy as np

def canary_decision(stable_errors, canary_errors, is_canary,
                    max_ratio=1.15, min_requests=100):
    """Decide whether to promote canary based on error rate comparison."""
    canary_reqs = int(is_canary.sum())
    stable_reqs = int((~is_canary).sum())

    if stable_reqs == 0:
        return {"decision": "HOLD", "reason": "No stable traffic"}

    stable_rate = stable_errors[~is_canary].mean()
    canary_rate = canary_errors[is_canary].mean() if canary_reqs > 0 else 0.0

    ratio = canary_rate / stable_rate if stable_rate > 0 else float("inf")

    if canary_reqs < min_requests:
        return {
            "decision": "HOLD",
            "canary_rate": float(canary_rate),
            "stable_rate": float(stable_rate),
            "ratio": float(ratio),
            "canary_requests": canary_reqs,
            "reason": f"Insufficient samples: {canary_reqs} < {min_requests}",
        }

    if ratio <= max_ratio:
        return {
            "decision": "PROMOTE",
            "canary_rate": float(canary_rate),
            "stable_rate": float(stable_rate),
            "ratio": float(ratio),
            "canary_requests": canary_reqs,
            "reason": f"Ratio {ratio:.3f} <= {max_ratio}",
        }
    else:
        return {
            "decision": "ROLLBACK",
            "canary_rate": float(canary_rate),
            "stable_rate": float(stable_rate),
            "ratio": float(ratio),
            "canary_requests": canary_reqs,
            "reason": f"Ratio {ratio:.3f} > {max_ratio}",
        }

rng = np.random.default_rng(42)
N = 2000

stable_err_1 = rng.random(N) < 0.035
canary_err_1 = rng.random(N) < 0.038
is_canary_1 = rng.random(N) < 0.10

stable_err_2 = rng.random(N) < 0.035
canary_err_2 = rng.random(N) < 0.055
is_canary_2 = rng.random(N) < 0.10

stable_err_3 = rng.random(N) < 0.035
canary_err_3 = rng.random(N) < 0.037
is_canary_3 = rng.random(N) < 0.02

for i, (se, ce, ic) in enumerate([
    (stable_err_1, canary_err_1, is_canary_1),
    (stable_err_2, canary_err_2, is_canary_2),
    (stable_err_3, canary_err_3, is_canary_3),
], 1):
    result = canary_decision(se, ce, ic)
    print(f"Scenario {i}: {result['decision']} - {result['reason']}")

**Exercise 6: PSI Calculation from Scratch (Coding).** Implement Population Stability Index with edge case handling.

In [ ]:
import numpy as np
import pandas as pd

def calculate_psi(expected, actual, bins=10, epsilon=1e-6):
    """Calculate PSI with proper edge case handling."""
    # Bin edges from expected (training) distribution
    quantiles = np.linspace(0, 100, bins + 1)
    edges = np.percentile(expected, quantiles)
    edges[0] = -np.inf
    edges[-1] = np.inf

    # Histogram counts
    expected_counts, _ = np.histogram(expected, bins=edges)
    actual_counts, _ = np.histogram(actual, bins=edges)

    # Convert to proportions with epsilon
    expected_pct = expected_counts / len(expected) + epsilon
    actual_pct = actual_counts / len(actual) + epsilon

    # PSI calculation
    psi_contributions = (actual_pct - expected_pct) * np.log(actual_pct / expected_pct)
    psi_score = float(psi_contributions.sum())

    # Interpretation
    if psi_score < 0.1:
        interpretation = "stable"
    elif psi_score < 0.25:
        interpretation = "moderate shift"
    else:
        interpretation = "major shift"

    details = pd.DataFrame({
        "bin_id": range(len(psi_contributions)),
        "expected_pct": expected_pct - epsilon,
        "actual_pct": actual_pct - epsilon,
        "psi_contribution": psi_contributions,
    })

    return psi_score, details, interpretation

rng = np.random.default_rng(123)
training = rng.normal(45, 12, 10000)
stable = rng.normal(45, 12, 2000)
moderate = rng.normal(48, 14, 2000)
major = rng.normal(60, 20, 2000)

for name, data in [("stable", stable), ("moderate", moderate), ("major", major)]:
    psi, details, interp = calculate_psi(training, data)
    print(f"{name:>10}: PSI={psi:.4f}, interpretation='{interp}'")
    print(details.to_string(index=False))
    print()

**Exercise 7: Multi-Feature PSI Monitor (Coding).** Track PSI across multiple features and prioritize alerts.

In [ ]:
import numpy as np
import pandas as pd

def calculate_psi_simple(expected, actual, bins=10, epsilon=1e-6):
    """Simplified PSI for a single feature."""
    edges = np.percentile(expected, np.linspace(0, 100, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    e_counts, _ = np.histogram(expected, bins=edges)
    a_counts, _ = np.histogram(actual, bins=edges)
    e_pct = e_counts / len(expected) + epsilon
    a_pct = a_counts / len(actual) + epsilon
    return float(((a_pct - e_pct) * np.log(a_pct / e_pct)).sum())

def monitor_drift(training_data, live_data, feature_importance):
    """Calculate weighted PSI across features."""
    results = []
    for col in training_data.select_dtypes(include=[np.number]).columns:
        psi = calculate_psi_simple(training_data[col].values, live_data[col].values)
        imp = feature_importance.get(col, 0.0)
        priority = psi * imp

        if psi < 0.1:
            status = "ok"
        elif psi < 0.25:
            status = "watch"
        else:
            status = "alert"

        results.append({
            "feature": col, "psi": round(psi, 4),
            "importance": imp, "alert_priority": round(priority, 4),
            "status": status,
        })

    return pd.DataFrame(results).sort_values("alert_priority", ascending=False).reset_index(drop=True)

rng = np.random.default_rng(456)
train_df = pd.DataFrame({
    "age": rng.normal(35, 10, 5000),
    "income": rng.normal(50000, 15000, 5000),
    "credit_score": rng.normal(680, 80, 5000),
    "loan_amount": rng.normal(25000, 10000, 5000),
})
live_df = pd.DataFrame({
    "age": rng.normal(35.5, 10.5, 1000),
    "income": rng.normal(53000, 18000, 1000),
    "credit_score": rng.normal(650, 90, 1000),
    "loan_amount": rng.normal(26000, 11000, 1000),
})
importance = {"credit_score": 0.45, "income": 0.30, "loan_amount": 0.15, "age": 0.10}

result = monitor_drift(train_df, live_df, importance)
print(result.to_string(index=False))

**Exercise 8: Alert Routing System (Coding).** Route monitoring signals to appropriate channels.

In [ ]:
def route_alert(metric_name, value, thresholds):
    """Route an alert based on metric thresholds."""
    config = thresholds[metric_name]
    is_upper = config["type"] == "upper"

    if is_upper:
        if value >= config["page"]:
            return {"route": "page", "priority": "P0",
                    "message": f"CRITICAL: {metric_name}={value} >= {config['page']}",
                    "actionable": True}
        elif value >= config["ticket"]:
            return {"route": "ticket", "priority": "P1",
                    "message": f"WARNING: {metric_name}={value} >= {config['ticket']}",
                    "actionable": True}
        else:
            return {"route": "digest", "priority": "P2",
                    "message": f"OK: {metric_name}={value}",
                    "actionable": False}
    else:  # lower threshold (e.g., daily_predictions)
        if value <= config["page"]:
            return {"route": "page", "priority": "P0",
                    "message": f"CRITICAL: {metric_name}={value} <= {config['page']}",
                    "actionable": True}
        elif value <= config["ticket"]:
            return {"route": "ticket", "priority": "P1",
                    "message": f"WARNING: {metric_name}={value} <= {config['ticket']}",
                    "actionable": True}
        else:
            return {"route": "digest", "priority": "P2",
                    "message": f"OK: {metric_name}={value}",
                    "actionable": False}

thresholds = {
    "error_rate": {"page": 0.05, "ticket": 0.02, "type": "upper"},
    "psi_top_feature": {"page": 0.25, "ticket": 0.10, "type": "upper"},
    "latency_p95": {"page": 500, "ticket": 300, "type": "upper"},
    "data_freshness_hours": {"page": 4, "ticket": 2, "type": "upper"},
    "daily_predictions": {"page": 1000, "ticket": 5000, "type": "lower"},
}

events = [
    ("error_rate", 0.08),
    ("psi_top_feature", 0.15),
    ("latency_p95", 450),
    ("data_freshness_hours", 5.5),
    ("daily_predictions", 800),
    ("error_rate", 0.015),
]

for metric, value in events:
    result = route_alert(metric, value, thresholds)
    print(f"{metric}={value}: [{result['priority']}] -> {result['route']}")
    print(f"  {result['message']}")
    print()

**Exercise 9: Alert Fatigue Analysis (Coding).** Analyze monitoring data for fatigue patterns.

In [ ]:
import numpy as np
import pandas as pd

def analyze_alert_fatigue(alerts_df):
    """Analyze alert fatigue patterns."""
    pages = alerts_df[alerts_df["route"] == "page"]

    total_pages = len(pages)
    ack_rate = pages["acknowledged"].mean() if total_pages > 0 else 0
    actionable_rate = pages["actionable"].mean() if total_pages > 0 else 0

    # False positive rate by metric (pages sent but not actionable)
    page_by_metric = pages.groupby("metric").agg(
        total=("actionable", "size"),
        not_actionable=("actionable", lambda x: (~x).sum()),
    )
    page_by_metric["false_positive_rate"] = (
        page_by_metric["not_actionable"] / page_by_metric["total"]
    )
    worst_offenders = (
        page_by_metric["false_positive_rate"]
        .sort_values(ascending=False)
        .to_dict()
    )

    recommendations = []
    for metric, rate in worst_offenders.items():
        if rate > 0.5:
            recommendations.append(f"{metric}: false positive rate {rate:.0%} — tighten threshold")

    if actionable_rate < 0.5:
        recommendations.append("Overall actionable rate low — review all thresholds")

    return {
        "total_pages": total_pages,
        "ack_rate": round(float(ack_rate), 3),
        "actionable_rate": round(float(actionable_rate), 3),
        "worst_offenders": worst_offenders,
        "recommendations": recommendations,
    }

rng = np.random.default_rng(789)
dates = pd.date_range("2026-07-01", "2026-07-30", freq="h")
n = len(dates)
alerts_df = pd.DataFrame({
    "timestamp": dates,
    "metric": rng.choice(["error_rate", "psi", "latency", "freshness"], n),
    "value": rng.uniform(0, 1, n),
    "route": rng.choice(["page", "ticket", "digest"], n, p=[0.15, 0.35, 0.50]),
    "acknowledged": rng.choice([True, False], n, p=[0.7, 0.3]),
    "actionable": rng.choice([True, False], n, p=[0.6, 0.4]),
})

report = analyze_alert_fatigue(alerts_df)
print(f"Total pages: {report['total_pages']}")
print(f"Acknowledgment rate: {report['ack_rate']:.1%}")
print(f"Actionable rate: {report['actionable_rate']:.1%}")
print(f"\nWorst false positive rates:")
for metric, rate in report["worst_offenders"].items():
    print(f"  {metric}: {rate:.1%}")
print(f"\nRecommendations:")
for rec in report["recommendations"]:
    print(f"  - {rec}")

**Exercise 10: End-to-End Gate Simulation (Challenge).** Build a complete deployment pipeline simulation.

In [ ]:
import numpy as np

def calculate_psi_simple(expected, actual, bins=10, epsilon=1e-6):
    edges = np.percentile(expected, np.linspace(0, 100, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    e_counts, _ = np.histogram(expected, bins=edges)
    a_counts, _ = np.histogram(actual, bins=edges)
    e_pct = e_counts / len(expected) + epsilon
    a_pct = a_counts / len(actual) + epsilon
    return float(((a_pct - e_pct) * np.log(a_pct / e_pct)).sum())

def deploy_pipeline(tests_pass, data_valid, accuracy, training_dist,
                    stable_error_rate, canary_error_rate, canary_input_dist,
                    min_accuracy=0.88, max_error_ratio=1.15, max_psi=0.25):
    """Simulate full deployment pipeline with 3 stages."""
    details = {}

    # Stage 1: CI Gates
    stage1_issues = []
    if not tests_pass:
        stage1_issues.append("tests failed")
    if not data_valid:
        stage1_issues.append("data validation failed")
    if accuracy < min_accuracy:
        stage1_issues.append(f"accuracy {accuracy:.3f} < {min_accuracy}")

    if stage1_issues:
        return {
            "stage": "CI",
            "decision": "ROLLBACK",
            "details": {"issues": stage1_issues, "accuracy": accuracy},
            "final_status": "rejected_at_ci",
        }

    details["ci"] = "passed"

    # Stage 2: Canary error rate check
    error_ratio = canary_error_rate / stable_error_rate if stable_error_rate > 0 else float("inf")
    if error_ratio > max_error_ratio:
        return {
            "stage": "canary",
            "decision": "ROLLBACK",
            "details": {"ci": "passed", "error_ratio": round(error_ratio, 3)},
            "final_status": "rejected_at_canary",
        }

    details["canary"] = {"error_ratio": round(error_ratio, 3), "status": "passed"}

    # Stage 3: PSI check
    psi = calculate_psi_simple(training_dist, canary_input_dist)
    if psi > max_psi:
        return {
            "stage": "drift",
            "decision": "ROLLBACK",
            "details": {"ci": "passed", "canary": details["canary"], "psi": round(psi, 4)},
            "final_status": "rejected_at_drift",
        }

    details["drift"] = {"psi": round(psi, 4), "status": "passed"}
    return {
        "stage": "all",
        "decision": "PROMOTE",
        "details": details,
        "final_status": "promoted_to_100",
    }

rng = np.random.default_rng(101)
training = rng.normal(0, 1, 10000)
canary_input = rng.normal(0.05, 1.05, 500)

scenarios = [
    ("Perfect deployment", True, True, 0.92, 0.03, 0.032, canary_input),
    ("Failed CI: accuracy too low", True, True, 0.80, 0.03, 0.032, canary_input),
    ("Failed canary: PSI too high", True, True, 0.91, 0.03, 0.032, rng.normal(1.5, 0.5, 500)),
]

for name, tp, dv, acc, ser, cer, cid in scenarios:
    result = deploy_pipeline(tp, dv, acc, training, ser, cer, cid)
    print(f"{name}:")
    print(f"  Stage: {result['stage']}, Decision: {result['decision']}")
    print(f"  Status: {result['final_status']}")
    print()

**Exercise 11: CT Trigger Logic (Challenge).** Design a Continuous Training trigger system.

In [ ]:
def should_retrain(days_since_train, psi_scores, performance_drop,
                   feature_importance, schedule_day, force=False):
    """Decide whether to trigger retraining based on multiple signals."""
    if force:
        return True, "Manual override: force=True"

    if schedule_day == "Monday":
        return True, "Scheduled weekly retraining (Monday)"

    # High-importance feature with high PSI
    for feat, imp in feature_importance.items():
        if imp > 0.3 and psi_scores.get(feat, 0) > 0.25:
            return True, f"High-importance feature '{feat}' (imp={imp:.2f}) has PSI={psi_scores[feat]:.3f} > 0.25"

    # Performance drop + any feature PSI elevated
    if performance_drop > 0.05 and any(v > 0.15 for v in psi_scores.values()):
        return True, f"Performance dropped {performance_drop:.1%} and drift detected"

    # Staleness check
    if days_since_train > 30:
        weighted_psi = sum(psi_scores.get(f, 0) * feature_importance.get(f, 0)
                           for f in feature_importance)
        if weighted_psi > 0.15:
            return True, f"Model stale ({days_since_train}d) with weighted PSI={weighted_psi:.3f}"

    return False, "No trigger conditions met"

feature_importance = {
    "credit_score": 0.45, "income": 0.30,
    "debt_ratio": 0.15, "age": 0.10,
}

scenarios = [
    (7, {"credit_score": 0.05, "income": 0.08, "debt_ratio": 0.03, "age": 0.04}, 0.01, "Monday", False),
    (15, {"credit_score": 0.28, "income": 0.12, "debt_ratio": 0.09, "age": 0.05}, 0.02, "Wednesday", False),
    (12, {"credit_score": 0.18, "income": 0.16, "debt_ratio": 0.11, "age": 0.08}, 0.07, "Friday", False),
    (35, {"credit_score": 0.14, "income": 0.17, "debt_ratio": 0.13, "age": 0.19}, 0.03, "Tuesday", False),
    (3, {"credit_score": 0.02, "income": 0.03, "debt_ratio": 0.01, "age": 0.02}, 0.00, "Thursday", True),
]

for i, (days, psi, perf, day, force) in enumerate(scenarios, 1):
    decision, reason = should_retrain(days, psi, perf, feature_importance, day, force)
    print(f"Scenario {i}: {'RETRAIN' if decision else 'HOLD'} - {reason}")

**Exercise 12: Monitoring Dashboard Simulator (Challenge).** Generate audience-appropriate monitoring reports.

In [ ]:
import numpy as np
import pandas as pd

def generate_monitoring_report(metrics_df, audience="engineer"):
    """Generate monitoring report for specified audience."""
    error_rate = metrics_df["errors"].sum() / metrics_df["requests"].sum()
    avg_latency_95 = metrics_df["latency_p95"].mean()
    avg_accuracy = metrics_df["accuracy"].mean()
    max_psi = metrics_df["psi_max"].max()
    days = len(metrics_df)
    date_range = f"{metrics_df['timestamp'].min().date()} to {metrics_df['timestamp'].max().date()}"

    if audience == "executive":
        lines = [
            f"=== EXECUTIVE MONITORING REPORT ===",
            f"Period: {date_range} ({days} days)\n",
            f"Key Performance Indicators:",
            f"  Model accuracy:     {avg_accuracy:.1%}{'  [OK]' if avg_accuracy >= 0.88 else '  [ALERT]'}",
            f"  Error rate:         {error_rate:.2%}{'  [OK]' if error_rate < 0.05 else '  [ALERT]'}",
            f"  P95 latency:        {avg_latency_95:.0f}ms{'  [OK]' if avg_latency_95 < 200 else '  [ALERT]'}",
            f"  Data drift (max):   {max_psi:.3f}{'  [OK]' if max_psi < 0.1 else '  [WATCH]' if max_psi < 0.25 else '  [ALERT]'}",
        ]
        if max_psi >= 0.25:
            lines.append(f"\nAction Required: Investigate data drift. Consider retraining.")
    else:
        lines = [
            f"=== ENGINEER MONITORING REPORT ===",
            f"Period: {date_range} ({days} days)\n",
            f"Throughput:",
            f"  Total requests:     {metrics_df['requests'].sum():,}",
            f"  Daily avg:          {metrics_df['requests'].mean():,.0f}",
            f"  Total errors:       {metrics_df['errors'].sum():,}\n",
            f"Latency:",
            f"  P95 avg:            {avg_latency_95:.1f}ms",
            f"  P95 min:            {metrics_df['latency_p95'].min():.1f}ms",
            f"  P95 max:            {metrics_df['latency_p95'].max():.1f}ms",
            f"  P99 avg:            {metrics_df['latency_p99'].mean():.1f}ms\n",
            f"Model Quality:",
            f"  Accuracy avg:       {avg_accuracy:.4f}",
            f"  Accuracy min:       {metrics_df['accuracy'].min():.4f}\n",
            f"Drift:",
            f"  PSI max:            {max_psi:.4f}",
            f"  PSI avg:            {metrics_df['psi_max'].mean():.4f}\n",
            f"Action Items:",
        ]
        if max_psi >= 0.25:
            lines.append(f"  - [URGENT] PSI={max_psi:.3f}: investigate distribution shift")
        if avg_accuracy < 0.88:
            lines.append(f"  - [URGENT] Accuracy={avg_accuracy:.3f} below 0.88 threshold")
        if error_rate >= 0.05:
            lines.append(f"  - [URGENT] Error rate={error_rate:.2%} above 5% threshold")
        if all(v < 0.1 for v in [max_psi]) and avg_accuracy >= 0.88:
            lines.append(f"  - None. System operating within bounds.")

    return "\n".join(lines)

rng = np.random.default_rng(999)
dates = pd.date_range("2026-08-01", "2026-08-26", freq="D")
metrics_df = pd.DataFrame({
    "timestamp": dates,
    "requests": rng.integers(8000, 12000, len(dates)),
    "errors": rng.integers(50, 200, len(dates)),
    "latency_p95": rng.uniform(120, 180, len(dates)),
    "latency_p99": rng.uniform(200, 300, len(dates)),
    "accuracy": rng.uniform(0.88, 0.93, len(dates)),
    "psi_max": rng.uniform(0.02, 0.30, len(dates)),
})

print(generate_monitoring_report(metrics_df, "executive"))
print("\n" + "=" * 50 + "\n")
print(generate_monitoring_report(metrics_df, "engineer"))